In [ ]:
"""Q Actor-Critic (QAC) for Acrobot-v1.

Performance metrics: training return and 50-episode moving-average return;
episode length; checkpoint and final greedy success rates; final greedy mean/std
return and mean steps to goal; sample efficiency; critic TD loss, mean absolute
TD error, policy entropy, and wall-clock training time.
"""

In [5]:
import os
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.distributions import Categorical

DEVICE = torch.device("cpu")
CONFIG = {
    "seed": 42, "hidden_size": 64, "actor_lr": 5e-4, "critic_lr": 1e-3,
    "gamma": 0.99, "entropy_coef": 0.01, "episodes": 1_000, "max_steps": 500,
    "eval_interval": 100, "eval_episodes": 10, "final_eval_episodes": 100,
    "ma_window": 50, "efficiency_threshold": -300.0,
}
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print(f"PyTorch {torch.__version__} | Gymnasium {gym.__version__} | {DEVICE}")

PyTorch 2.13.0+cpu | Gymnasium 1.3.0 | cpu


In [6]:
class MLP(nn.Module):
    def __init__(self, state_dim, output_dim, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, output_dim),
        )

    def forward(self, state):
        return self.net(state)


def build_agent(env, cfg):
    state_dim, action_dim = env.observation_space.shape[0], env.action_space.n
    actor = MLP(state_dim, action_dim, cfg["hidden_size"]).to(DEVICE)
    critic = MLP(state_dim, action_dim, cfg["hidden_size"]).to(DEVICE)
    return actor, critic, optim.Adam(actor.parameters(), lr=cfg["actor_lr"]), optim.Adam(critic.parameters(), lr=cfg["critic_lr"])


def run_episode(env, actor, critic, actor_opt=None, critic_opt=None, cfg=None, greedy=False):
    state, _ = env.reset()
    state = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
    reward_sum = loss_sum = td_sum = entropy_sum = 0.0
    updates = 0

    for length in range(1, cfg["max_steps"] + 1):
        if greedy:
            with torch.inference_mode():
                action = int(actor(state).argmax().item())
        else:
            distribution = Categorical(logits=actor(state))
            action_tensor = distribution.sample()
            action = int(action_tensor.item())

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = torch.as_tensor(next_state, dtype=torch.float32, device=DEVICE)
        reward_sum += reward

        if not greedy:
            log_prob, entropy = distribution.log_prob(action_tensor), distribution.entropy()
            with torch.no_grad():
                target = torch.as_tensor(reward, dtype=torch.float32, device=DEVICE) if done else reward + cfg["gamma"] * critic(next_state)[Categorical(logits=actor(next_state)).sample()]
            q_sa = critic(state)[action]
            td_error = target - q_sa
            critic_loss = td_error.square()
            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()

            actor_loss = -(log_prob * q_sa.detach()) - cfg["entropy_coef"] * entropy
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()
            loss_sum += critic_loss.item()
            td_sum += td_error.abs().item()
            entropy_sum += entropy.item()
            updates += 1

        if done:
            break
        state = next_state

    return {"reward": reward_sum, "length": length, "success": int(terminated),
            "critic_loss": loss_sum / updates if updates else np.nan,
            "mean_abs_td_error": td_sum / updates if updates else np.nan,
            "entropy": entropy_sum / updates if updates else np.nan}


def evaluate(env, actor, critic, cfg, episodes):
    results = [run_episode(env, actor, critic, cfg=cfg, greedy=True) for _ in range(episodes)]
    return {key: np.array([row[key] for row in results]) for key in ("reward", "length", "success")}

In [3]:
def train(env, eval_env, actor, critic, actor_opt, critic_opt, cfg):
    history = {key: [] for key in ("reward", "length", "success", "critic_loss", "mean_abs_td_error", "entropy")}
    checkpoints = []
    start = time.perf_counter()
    for episode in range(1, cfg["episodes"] + 1):
        result = run_episode(env, actor, critic, actor_opt, critic_opt, cfg)
        for key in history:
            history[key].append(result[key])
        if episode % cfg["eval_interval"] == 0:
            evaluation = evaluate(eval_env, actor, critic, cfg, cfg["eval_episodes"])
            rate = 100 * evaluation["success"].mean()
            checkpoints.append((episode, rate, evaluation["reward"].mean()))
            print(f"Episode {episode:4d} | MA return: {np.mean(history['reward'][-cfg['ma_window']:]):7.1f} | Eval success: {rate:5.1f}%")
    history["training_time_sec"] = time.perf_counter() - start
    history["checkpoints"] = checkpoints
    return history


def summarize(history, final_eval, cfg):
    rewards = np.asarray(history["reward"])
    moving_average = np.convolve(rewards, np.ones(cfg["ma_window"])/cfg["ma_window"], mode="valid")
    crossing = np.flatnonzero(moving_average >= cfg["efficiency_threshold"])
    successful_lengths = final_eval["length"][final_eval["success"].astype(bool)]
    return {
        "training_episodes": cfg["episodes"], "best_ma_return": moving_average.max(),
        "sample_efficiency_episode": int(crossing[0] + cfg["ma_window"]) if crossing.size else np.nan,
        "final_eval_mean_return": final_eval["reward"].mean(), "final_eval_std_return": final_eval["reward"].std(),
        "final_eval_success_rate_pct": 100 * final_eval["success"].mean(),
        "final_eval_mean_length": final_eval["length"].mean(),
        "mean_steps_to_goal": successful_lengths.mean() if successful_lengths.size else np.nan,
        "training_time_sec": history["training_time_sec"],
        "final_critic_td_loss": np.nanmean(history["critic_loss"][-100:]),
        "final_mean_abs_td_error": np.nanmean(history["mean_abs_td_error"][-100:]),
        "final_policy_entropy": np.nanmean(history["entropy"][-100:]),
    }


def plot_results(history, cfg, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    rewards = np.asarray(history["reward"])
    figure, axes = plt.subplots(2, 2, figsize=(12, 7))
    window = cfg["ma_window"]
    axes[0, 0].plot(rewards, alpha=.25, label="return")
    axes[0, 0].plot(np.arange(window - 1, len(rewards)), np.convolve(rewards, np.ones(window)/window, mode="valid"), label=f"{window}-episode MA")
    axes[0, 0].set(title="Training return", xlabel="Episode", ylabel="Return")
    axes[0, 1].plot(history["length"], color="darkorange")
    axes[0, 1].set(title="Episode length", xlabel="Episode", ylabel="Steps")
    checkpoint_array = np.asarray(history["checkpoints"])
    axes[1, 0].plot(checkpoint_array[:, 0], checkpoint_array[:, 1], marker="o", color="seagreen")
    axes[1, 0].set(title="Greedy checkpoint success", xlabel="Training episode", ylabel="Success (%)", ylim=(0, 105))
    axes[1, 1].plot(history["critic_loss"], label="TD loss")
    axes[1, 1].plot(history["mean_abs_td_error"], label="|TD error|")
    axes[1, 1].set(title="Critic diagnostics", xlabel="Episode")
    for axis in axes.flat:
        axis.legend() if axis.get_legend_handles_labels()[0] else None
    figure.tight_layout()
    figure.savefig(os.path.join(output_dir, "qac_metrics_optimized.png"), dpi=150)
    plt.show()

In [4]:
env, eval_env = gym.make("Acrobot-v1"), gym.make("Acrobot-v1")
env.reset(seed=CONFIG["seed"])
eval_env.reset(seed=CONFIG["seed"] + 1)
actor, critic, actor_opt, critic_opt = build_agent(env, CONFIG)
history = train(env, eval_env, actor, critic, actor_opt, critic_opt, CONFIG)
final_eval = evaluate(eval_env, actor, critic, CONFIG, CONFIG["final_eval_episodes"])
metrics = summarize(history, final_eval, CONFIG)

output_dir = os.path.dirname(os.path.abspath("Q_Actor_Critic_Acrobot_Optimized.ipynb")) or "."
episode_history = {key: values for key, values in history.items() if isinstance(values, list) and key != "checkpoints"}
pd.DataFrame(episode_history).to_csv(os.path.join(output_dir, "results_qac_optimized.csv"), index_label="episode")
pd.DataFrame([metrics]).round(4).to_csv(os.path.join(output_dir, "summary_qac_optimized.csv"), index=False)
display(pd.DataFrame([metrics]).round(2).T.rename(columns={0: "value"}))
plot_results(history, CONFIG, os.path.join(output_dir, "figures"))
env.close()
eval_env.close()

Episode  100 | MA return:  -500.0 | Eval success:   0.0%
Episode  200 | MA return:  -500.0 | Eval success:   0.0%
Episode  300 | MA return:  -500.0 | Eval success:   0.0%
Episode  400 | MA return:  -500.0 | Eval success:   0.0%


KeyboardInterrupt: 